## Data Collection & Dataset Splitting

### Objectives
* Setup Kaggle authentication and download the official Cherry Leaves dataset from Kaggle.
* Inspect and clean dataset by removing non-image files.
* Split dataset randomly into **Train (70%)**, **Validation (10%)**, and **Test (20%)** sets.
* Plot and save the image distribution across splits to verify class balance.

### Inputs
* `kaggle.json` API authentication file located in project root.
* Kaggle Dataset: `codeinstitute/cherry-leaves`

### Outputs
* `inputs/cherry_leaves/` directory containing `train`, `validation`, and `test` folders split into `healthy` and `powdery_mildew`.
* `outputs/v1/class_distribution.png` visual plot.

In [5]:
import os
import glob
import random
import shutil
import zipfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# 1. Resolve Project Root Directory
current_dir = os.getcwd()
if os.path.basename(current_dir) == 'jupyter_notebooks':
    PROJECT_DIR = os.path.abspath(os.path.join(current_dir, '..'))
else:
    PROJECT_DIR = current_dir

# 2. Configure Kaggle Credentials Path
os.environ['KAGGLE_CONFIG_DIR'] = PROJECT_DIR

# 3. Define Project Subdirectories
INPUTS_DIR = os.path.join(PROJECT_DIR, 'inputs')
RAW_DATA_DIR = os.path.join(INPUTS_DIR, 'raw_data')
SPLIT_DATA_DIR = os.path.join(INPUTS_DIR, 'cherry_leaves')
OUTPUTS_DIR = os.path.join(PROJECT_DIR, 'outputs', 'v1')

# 4. Create Directories at Project Root
os.makedirs(RAW_DATA_DIR, exist_ok=True)
os.makedirs(SPLIT_DATA_DIR, exist_ok=True)
os.makedirs(OUTPUTS_DIR, exist_ok=True)

print(f"Project Root: {PROJECT_DIR}")
print(f"Outputs Directory: {OUTPUTS_DIR}")

Project Root: d:\books\code\code-institut\PROJECTS\cherry tree leaves
Outputs Directory: d:\books\code\code-institut\PROJECTS\cherry tree leaves\outputs\v1


In [6]:
import os

# If current directory is 'jupyter_notebooks', navigate one level up to project root
current_dir = os.getcwd()
if os.path.basename(current_dir) == 'jupyter_notebooks':
    PROJECT_DIR = os.path.abspath(os.path.join(current_dir, '..'))
else:
    PROJECT_DIR = current_dir

INPUTS_DIR = os.path.join(PROJECT_DIR, 'inputs')
RAW_DATA_DIR = os.path.join(INPUTS_DIR, 'raw_data')
SPLIT_DATA_DIR = os.path.join(INPUTS_DIR, 'cherry_leaves')
OUTPUTS_DIR = os.path.join(PROJECT_DIR, 'outputs', 'v1')

print(f"Project Root Directory: {PROJECT_DIR}")

Project Root Directory: d:\books\code\code-institut\PROJECTS\cherry tree leaves


In [7]:
%pip install kaggle

Note: you may need to restart the kernel to use updated packages.


In [8]:
# Download Kaggle Dataset
import kaggle

dataset_name = "codeinstitute/cherry-leaves"
print(f"Downloading dataset '{dataset_name}'...")

kaggle.api.dataset_download_files(dataset_name, path=RAW_DATA_DIR, unzip=True)
print("Download and extraction complete.")

Dataset URL: https://www.kaggle.com/datasets/codeinstitute/cherry-leaves
Download and extraction complete.


In [9]:
# Verify and remove non-image files
valid_extensions = ('.png', '.jpg', '.jpeg', '.PNG', '.JPG', '.JPEG')
removed_count = 0

for root, _, files in os.walk(RAW_DATA_DIR):
    for file in files:
        file_path = os.path.join(root, file)
        if not file.endswith(valid_extensions):
            os.remove(file_path)
            removed_count += 1
            print(f"Removed non-image file: {file_path}")

print(f"Data cleaning complete. Total invalid files removed: {removed_count}")

Data cleaning complete. Total invalid files removed: 0


In [11]:
import os
import random
import shutil
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seed for reproducibility
random.seed(42)

# Define split ratios
train_ratio, val_ratio, test_ratio = 0.70, 0.10, 0.20
labels = ['healthy', 'powdery_mildew']
splits = ['train', 'validation', 'test']

# 1. Create split subdirectories
for split in splits:
    for label in labels:
        os.makedirs(os.path.join(SPLIT_DATA_DIR, split, label), exist_ok=True)

summary_data = []

# 2. Dynamically locate label directories and split dataset
for label in labels:
    # Recursively search RAW_DATA_DIR for the matching class folder
    label_dir = None
    for root, dirs, files in os.walk(RAW_DATA_DIR):
        if os.path.basename(root).lower() == label.lower():
            label_dir = root
            break

    # Guard clause if raw dataset was not downloaded or found
    if not label_dir or not os.path.exists(label_dir):
        raise FileNotFoundError(
            f"Could not locate folder for '{label}' in '{RAW_DATA_DIR}'. "
            f"Please verify Cell 5 (Kaggle download) completed successfully."
        )

    images = [f for f in os.listdir(label_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    random.shuffle(images)

    total_imgs = len(images)
    train_count = int(total_imgs * train_ratio)
    val_count = int(total_imgs * val_ratio)

    train_imgs = images[:train_count]
    val_imgs = images[train_count:train_count + val_count]
    test_imgs = images[train_count + val_count:]

    # Copy files to respective split directories
    for img in train_imgs:
        shutil.copy(os.path.join(label_dir, img), os.path.join(SPLIT_DATA_DIR, 'train', label, img))
    for img in val_imgs:
        shutil.copy(os.path.join(label_dir, img), os.path.join(SPLIT_DATA_DIR, 'validation', label, img))
    for img in test_imgs:
        shutil.copy(os.path.join(label_dir, img), os.path.join(SPLIT_DATA_DIR, 'test', label, img))

    print(f"Category '{label}': {total_imgs} total -> Train: {len(train_imgs)}, Val: {len(val_imgs)}, Test: {len(test_imgs)}")

    summary_data.extend([
        {'Split': 'Train', 'Label': label.replace('_', ' ').title(), 'Count': len(train_imgs)},
        {'Split': 'Validation', 'Label': label.replace('_', ' ').title(), 'Count': len(val_imgs)},
        {'Split': 'Test', 'Label': label.replace('_', ' ').title(), 'Count': len(test_imgs)}
    ])

# 3. Print Summary DataFrame
df_summary = pd.DataFrame(summary_data)
print("\nDataset Distribution Summary:\n", df_summary)

# 4. Plot and Save Distribution Plot
plt.figure(figsize=(8, 5))
sns.barplot(data=df_summary, x='Split', y='Count', hue='Label')
plt.title('Image Distribution Across Dataset Splits')
plt.ylabel('Number of Images')

plot_path = os.path.join(OUTPUTS_DIR, 'class_distribution.png')
plt.savefig(plot_path, bbox_inches='tight')
plt.close()
print(f"\nDistribution plot saved to: {plot_path}")

Category 'healthy': 2104 total -> Train: 1472, Val: 210, Test: 422
Category 'powdery_mildew': 2104 total -> Train: 1472, Val: 210, Test: 422

Dataset Distribution Summary:
         Split           Label  Count
0       Train         Healthy   1472
1  Validation         Healthy    210
2        Test         Healthy    422
3       Train  Powdery Mildew   1472
4  Validation  Powdery Mildew    210
5        Test  Powdery Mildew    422

Distribution plot saved to: d:\books\code\code-institut\PROJECTS\cherry tree leaves\outputs\v1\class_distribution.png
